This code is used to compute the stock driven dynamic MFA model for the secondary melting capacity. It is splitted into a model for each melting type (remelter, refiner, foundry)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from dynamic_stock_model import DynamicStockModel

# Scenarios

In [6]:
# 'no change' = same inflow shares of technologies as today
# 'high electricity' = only electrified technologies from 2023 onwards (in inflows)
# 'electricity, hydrogen2030' = until 2030 electricity, then 50/50 in inflows
# refiner_scenario_H2 = H2 for Refiner from 2030 onwards, rest still fossil. For remelter and foundry high electricity'
scenario_tech_name = 'electricity, hydrogen2030'

#here, it can be changed if we read the secondary produciton without  ("S-") or with improved sorting and recycling technologies ("S+")
#also the suffix 'Highest improvement' can be added for the scenario with increasing EOL scrap collection rates, yields and advanced sorting and recycling technologies
scenario_name= scenario_tech_name + ', ' + ''

#here, te file that is used as an input can be selected:
#Secondary Melting_High demand_S+
#Secondary Melting_High demand_S+
#Secondary Melting_High demand_Highest improvement
#Secondary Melting_S-
#Secondary Melting_S+
#Secondary Melting_Highest improvement
file = "Technology stocks/Secondary melting/Secondary Melting_high demand_S+.xlsx"


#lifetime (20 (7 std) vs. 30 (10 std))
lifetime_furnaces = 30
std_dev_furnaces = 10
scenario_name

'electricity, hydrogen2030, '

# Remelter

In [7]:
#loading data_rem
data_rem = pd.read_excel(file, sheet_name = 'Remelter', usecols='A:AA', skiprows=1, nrows=151)
#Input data_rem (smelter capacity stocks) stays the same
data_rem.reset_index(drop=True, inplace=True)
data_rem.fillna(0, inplace=True)
data_rem

,year,Net capacity remelter [kt],Inflow remelter,Outflow remelter,Inflow share reverberatory,Inflow share induction,Inflow share hydrogen,Net capacity reverberatory,Stock change reverberatory,Inflow reverberatory,...,Energy demand induction,GHG emissions induction,Net capacity hydrogen,Stock change hydrogen,Inflow hydrogen,Outflow hydrogen,Energy demand hydrogen,GHG emissions hydrogen,Energy demand total [GWh],GHG emissions total [kt] BAU
0,1900,0.000000,0.000000,0.000000,0.9,0.1,0,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0,0.000000,0.0,0.0,0.0,0.0
1,1901,0.071295,0.071295,0.000000,0.9,0.1,0,0.064166,0.064166,0.064166,...,0.0,0.0,0.007130,0.007130,0,-0.007130,0.0,0.0,0.0,0.0
2,1902,0.085518,0.014325,0.000102,0.9,0.1,0,0.076966,0.012800,0.012892,...,0.0,0.0,0.001432,-0.005697,0,0.005697,0.0,0.0,0.0,0.0
3,1903,0.102349,0.017001,0.000169,0.9,0.1,0,0.092114,0.015148,0.015301,...,0.0,0.0,0.001700,0.000268,0,-0.000268,0.0,0.0,0.0,0.0
4,1904,0.122273,0.020191,0.000267,0.9,0.1,0,0.110046,0.017932,0.018172,...,0.0,0.0,0.002019,0.000319,0,-0.000319,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,2046,108840.269056,7462.898802,3986.359431,0.9,0.1,0,97956.242151,3128.885435,6716.608922,...,0.0,0.0,746.289880,16.391746,0,-16.391746,0.0,0.0,0.0,0.0
147,2047,112352.233926,7626.338656,4114.373786,0.9,0.1,0,101117.010533,3160.768383,6863.704790,...,0.0,0.0,762.633866,16.343985,0,-16.343985,0.0,0.0,0.0,0.0
148,2048,115895.921291,7789.321849,4245.634485,0.9,0.1,0,104306.329162,3189.318628,7010.389665,...,0.0,0.0,778.932185,16.298319,0,-16.298319,0.0,0.0,0.0,0.0
149,2049,119467.882137,7952.556067,4380.595221,0.9,0.1,0,107521.093923,3214.764761,7157.300461,...,0.0,0.0,795.255607,16.323422,0,-16.323422,0.0,0.0,0.0,0.0


In [4]:
#Remelter DSM model


DSM_remelter = DynamicStockModel(t=data_rem['year'],
                         s=data_rem['Net capacity remelter [kt]'],
                         lt={'Type': 'Normal', 
                             'Mean': np.array([lifetime_furnaces]),
                             'StdDev': np.array([std_dev_furnaces]) 
                             }
                            )

DSM_remelter.compute_stock_driven_model() 
DSM_remelter.compute_outflow_total()

data_rem['Inflow remelter'] = DSM_remelter.i
data_rem['Outflow remelter'] = DSM_remelter.o

In [5]:
#calculating stock by cohort to calculate capacity per technology
stock_cohort = DSM_remelter.s_c
stock_cohort

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 7.12954598e-02, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 7.12530210e-02, 1.42648632e-02, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 1.12829271e-04, 2.25749876e-05, ...,
        3.52273442e+03, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.12829271e-04, 2.25749876e-05, ...,
        3.52063750e+03, 3.79184534e+03, 0.00000000e+00],
       [0.00000000e+00, 1.12829271e-04, 2.25749876e-05, ...,
        3.51784910e+03, 3.78958823e+03, 4.06689197e+03]])

In [6]:
#setting the inflow shares for 2023 onwards depending on scenario
if scenario_tech_name == 'no change':
    data_rem.loc[data_rem['year']>=2023, 'Inflow share reverberatory'] = data_rem['Inflow share reverberatory'][122]
    data_rem.loc[data_rem['year']>=2023, 'Inflow share induction'] = data_rem['Inflow share induction'][122]
    data_rem.loc[data_rem['year']>=2023, 'Inflow share hydrogen'] = data_rem['Inflow share hydrogen'][122]

elif scenario_tech_name == 'high electricity':
    data_rem.loc[data_rem['year']>=2023, 'Inflow share reverberatory'] = 0
    data_rem.loc[data_rem['year']>=2023, 'Inflow share induction'] = 1
    data_rem.loc[data_rem['year']>=2023, 'Inflow share hydrogen'] = 0
    
elif scenario_tech_name == 'electricity, hydrogen2030':
    data_rem.loc[data_rem['year']>=2023, 'Inflow share reverberatory'] = 0
    data_rem.loc[data_rem['year']>=2023, 'Inflow share induction'] = 1
    data_rem.loc[data_rem['year']>=2023, 'Inflow share hydrogen'] = 0
    
    data_rem.loc[data_rem['year']>=2030, 'Inflow share induction'] = 0.5
    data_rem.loc[data_rem['year']>=2030, 'Inflow share hydrogen'] = 0.5

inflow_share_rev=np.array([data_rem['Inflow share reverberatory']])
inflow_share_induc=np.array([data_rem['Inflow share induction']])
inflow_share_h2=np.array([data_rem['Inflow share hydrogen']])

In [7]:
s_c_rev = stock_cohort*inflow_share_rev
s_c_induc = stock_cohort*inflow_share_induc
s_c_h2 = stock_cohort*inflow_share_h2

#np.sum([[0, 1], [0, 5]], axis=0)
#array([0, 6])
data_rem['Net capacity reverberatory'] = np.sum(s_c_rev, axis=1)
data_rem['Net capacity induction'] = np.sum(s_c_induc, axis=1)
data_rem['Net capacity hydrogen'] = np.sum(s_c_h2, axis=1)

In [8]:
data_rem

,year,Net capacity remelter [kt],Inflow remelter,Outflow remelter,Inflow share reverberatory,Inflow share induction,Inflow share hydrogen,Net capacity reverberatory,Stock change reverberatory,Inflow reverberatory,...,Energy demand induction,GHG emissions induction,Net capacity hydrogen,Stock change hydrogen,Inflow hydrogen,Outflow hydrogen,Energy demand hydrogen,GHG emissions hydrogen,Energy demand total [GWh],GHG emissions total [kt] BAU
0,1900,0.000000,0.000000,0.000000,0.9,0.1,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0,0.000000,0.0,0.0,0.0,0.0
1,1901,0.071295,0.071295,0.000000,0.9,0.1,0.0,0.064166,0.064166,0.064166,...,0.0,0.0,0.000000,0.007130,0,-0.007130,0.0,0.0,0.0,0.0
2,1902,0.085518,0.014265,0.000042,0.9,0.1,0.0,0.076966,0.012800,0.012838,...,0.0,0.0,0.000000,-0.005703,0,0.005703,0.0,0.0,0.0,0.0
3,1903,0.102349,0.016896,0.000065,0.9,0.1,0.0,0.092114,0.015148,0.015206,...,0.0,0.0,0.000000,0.000263,0,-0.000263,0.0,0.0,0.0,0.0
4,1904,0.122273,0.020020,0.000096,0.9,0.1,0.0,0.110046,0.017932,0.018018,...,0.0,0.0,0.000000,0.000312,0,-0.000312,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,2046,66675.513384,3046.292329,1959.356330,0.0,0.5,0.5,15416.188540,3128.885435,5186.807262,...,0.0,0.0,18814.546625,12.220473,0,-12.220473,0.0,0.0,0.0,0.0
147,2047,67952.859256,3270.837947,1993.492076,0.0,0.5,0.5,14150.573571,3160.768383,5294.677236,...,0.0,0.0,20349.154622,11.985553,0,-11.985553,0.0,0.0,0.0,0.0
148,2048,69451.588287,3522.734423,2024.005391,0.0,0.5,0.5,12922.554969,3189.318628,5400.025611,...,0.0,0.0,21989.840539,11.705375,0,-11.705375,0.0,0.0,0.0,0.0
149,2049,71192.409682,3791.845340,2051.023946,0.0,0.5,0.5,11738.809419,3214.764761,5503.092731,...,0.0,0.0,23742.452816,11.451902,0,-11.451902,0.0,0.0,0.0,0.0


In [9]:
data_rem['Inflow reverberatory']=DSM_remelter.i*data_rem['Inflow share reverberatory']
data_rem['Inflow induction']=DSM_remelter.i*data_rem['Inflow share induction']
data_rem['Inflow hydrogen']=DSM_remelter.i*data_rem['Inflow share hydrogen']

In [10]:
data_rem

,year,Net capacity remelter [kt],Inflow remelter,Outflow remelter,Inflow share reverberatory,Inflow share induction,Inflow share hydrogen,Net capacity reverberatory,Stock change reverberatory,Inflow reverberatory,...,Energy demand induction,GHG emissions induction,Net capacity hydrogen,Stock change hydrogen,Inflow hydrogen,Outflow hydrogen,Energy demand hydrogen,GHG emissions hydrogen,Energy demand total [GWh],GHG emissions total [kt] BAU
0,1900,0.000000,0.000000,0.000000,0.9,0.1,0.0,0.000000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,0.0,0.0,0.0
1,1901,0.071295,0.071295,0.000000,0.9,0.1,0.0,0.064166,0.064166,0.064166,...,0.0,0.0,0.000000,0.007130,0.000000,-0.007130,0.0,0.0,0.0,0.0
2,1902,0.085518,0.014265,0.000042,0.9,0.1,0.0,0.076966,0.012800,0.012838,...,0.0,0.0,0.000000,-0.005703,0.000000,0.005703,0.0,0.0,0.0,0.0
3,1903,0.102349,0.016896,0.000065,0.9,0.1,0.0,0.092114,0.015148,0.015206,...,0.0,0.0,0.000000,0.000263,0.000000,-0.000263,0.0,0.0,0.0,0.0
4,1904,0.122273,0.020020,0.000096,0.9,0.1,0.0,0.110046,0.017932,0.018018,...,0.0,0.0,0.000000,0.000312,0.000000,-0.000312,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,2046,66675.513384,3046.292329,1959.356330,0.0,0.5,0.5,15416.188540,3128.885435,0.000000,...,0.0,0.0,18814.546625,12.220473,1523.146165,-12.220473,0.0,0.0,0.0,0.0
147,2047,67952.859256,3270.837947,1993.492076,0.0,0.5,0.5,14150.573571,3160.768383,0.000000,...,0.0,0.0,20349.154622,11.985553,1635.418974,-11.985553,0.0,0.0,0.0,0.0
148,2048,69451.588287,3522.734423,2024.005391,0.0,0.5,0.5,12922.554969,3189.318628,0.000000,...,0.0,0.0,21989.840539,11.705375,1761.367211,-11.705375,0.0,0.0,0.0,0.0
149,2049,71192.409682,3791.845340,2051.023946,0.0,0.5,0.5,11738.809419,3214.764761,0.000000,...,0.0,0.0,23742.452816,11.451902,1895.922670,-11.451902,0.0,0.0,0.0,0.0


In [11]:
#export total data_rem
data_rem.to_excel('Python exports/Secondary_remelter' + '_' + scenario_name + ', L' + str(lifetime_furnaces) +'.xlsx')

# Refiner

In [12]:
#loading data_rem
data_ref = pd.read_excel(file, sheet_name = 'Refiner', usecols='A:AJ', skiprows=1, nrows=151)
#Input data_rem (smelter capacity stocks) stays the same
data_ref.reset_index(drop=True, inplace=True)
data_ref.fillna(0, inplace=True)
data_ref

,year,Net capacity refiner [kt],Inflow refiner,Outflow refiner,Inflow share rotary,Inflow share reverberatory,Inflow share induction,Inflow share hydrogen,Net capacity rotary,Stock change rotary,...,GHG emissions induction NZE [kt CO2 eq],Net capacity hydrogen,Stock change hydrogen,Inflow hydrogen,Outflow hydrogen,Energy demand hydrogen [GWh],GHG emissions hydrogen,Energy demand total [GWh],GHG emissions total [kt] BAU,GHG emissions total [kt] NZE
0,1900,0.000000,0.000000,0.000000,0.699807,0.300193,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1901,0.017407,0.017407,0.000000,0.699807,0.300193,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,1902,0.020760,0.003363,0.000010,0.699807,0.300193,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,1903,0.024733,0.003989,0.000016,0.699807,0.300193,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,1904,0.029447,0.004737,0.000023,0.699807,0.300193,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,2046,57242.165900,2713.893686,1383.634068,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
147,2047,58061.506652,2738.316923,1443.718562,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
148,2048,58878.520096,2761.635397,1504.356917,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
149,2049,59695.376698,2784.595753,1565.420358,0.000000,0.000000,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
#Refiner DSM model


DSM_refiner = DynamicStockModel(t=data_ref['year'],
                         s=data_ref['Net capacity refiner [kt]'],
                         lt={'Type': 'Normal', 
                             'Mean': np.array([lifetime_furnaces]),
                             'StdDev': np.array([std_dev_furnaces]) 
                             }
                            )

DSM_refiner.compute_stock_driven_model() 
DSM_refiner.compute_outflow_total()

data_ref['Inflow refiner'] = DSM_refiner.i
data_ref['Outflow refiner'] = DSM_refiner.o


In [14]:
#calculating stock by cohort to calculate capacity per technology
stock_cohort_ref = DSM_refiner.s_c
stock_cohort_ref

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.74074556e-02, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.73970937e-02, 3.36273561e-03, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 2.75483254e-05, 5.32172749e-06, ...,
        2.27758967e+03, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 2.75483254e-05, 5.32172749e-06, ...,
        2.27623393e+03, 2.32904282e+03, 0.00000000e+00],
       [0.00000000e+00, 2.75483254e-05, 5.32172749e-06, ...,
        2.27443111e+03, 2.32765645e+03, 2.38007125e+03]])

In [15]:
#setting the inflow shares for 2023 onwards depending on scenario
if scenario_tech_name == 'no change':
    data_ref.loc[data_ref['year']>=2023, 'Inflow share rotary'] = data_ref['Inflow share rotary'][122]
    data_ref.loc[data_ref['year']>=2023, 'Inflow share reverberatory'] = data_ref['Inflow share reverberatory'][122]
    data_ref.loc[data_ref['year']>=2023, 'Inflow share induction'] = data_ref['Inflow share induction'][122]
    data_ref.loc[data_ref['year']>=2023, 'Inflow share hydrogen'] = data_ref['Inflow share hydrogen'][122]

elif scenario_tech_name == 'high electricity':
    data_ref.loc[data_ref['year']>=2023, 'Inflow share rotary'] = 0
    data_ref.loc[data_ref['year']>=2023, 'Inflow share reverberatory'] = 0
    data_ref.loc[data_ref['year']>=2023, 'Inflow share induction'] = 1
    data_ref.loc[data_ref['year']>=2023, 'Inflow share hydrogen'] = 0
    
elif scenario_tech_name == 'electricity, hydrogen2030':
    data_ref.loc[data_ref['year']>=2023, 'Inflow share rotary'] = 0
    data_ref.loc[data_ref['year']>=2023, 'Inflow share reverberatory'] = 0
    data_ref.loc[data_ref['year']>=2023, 'Inflow share induction'] = 1
    data_ref.loc[data_ref['year']>=2023, 'Inflow share hydrogen'] = 0
    
    data_ref.loc[data_ref['year']>=2030, 'Inflow share induction'] = 0.5
    data_ref.loc[data_ref['year']>=2030, 'Inflow share hydrogen'] = 0.5
    
elif scenario_tech_name == 'refiner_scenario_H2':
    data_ref.loc[data_ref['year']>=2023, 'Inflow share rotary'] = data_ref['Inflow share rotary'][122]
    data_ref.loc[data_ref['year']>=2023, 'Inflow share reverberatory'] = data_ref['Inflow share reverberatory'][122]
    data_ref.loc[data_ref['year']>=2023, 'Inflow share induction'] = 0
    data_ref.loc[data_ref['year']>=2023, 'Inflow share hydrogen'] = 0
    
    data_ref.loc[data_ref['year']>=2030, 'Inflow share rotary'] = 0
    data_ref.loc[data_ref['year']>=2030, 'Inflow share reverberatory'] = 0
    data_ref.loc[data_ref['year']>=2030, 'Inflow share hydrogen'] = 1

inflow_share_ref_rot=np.array([data_ref['Inflow share rotary']])
inflow_share_ref_rev=np.array([data_ref['Inflow share reverberatory']])
inflow_share_ref_induc=np.array([data_ref['Inflow share induction']])
inflow_share_ref_h2=np.array([data_ref['Inflow share hydrogen']])

In [16]:
s_c_ref_rot = stock_cohort_ref*inflow_share_ref_rot
s_c_ref_rev = stock_cohort_ref*inflow_share_ref_rev
s_c_ref_induc = stock_cohort_ref*inflow_share_ref_induc
s_c_ref_h2 = stock_cohort_ref*inflow_share_ref_h2

#np.sum([[0, 1], [0, 5]], axis=0)
#array([0, 6])

data_ref['Net capacity rotary'] = np.sum(s_c_ref_rot, axis=1)
data_ref['Net capacity reverberatory'] = np.sum(s_c_ref_rev, axis=1)
data_ref['Net capacity induction'] = np.sum(s_c_ref_induc, axis=1)
data_ref['Net capacity hydrogen'] = np.sum(s_c_ref_h2, axis=1)

In [17]:
data_ref['Inflow rotary']=DSM_refiner.i*data_ref['Inflow share rotary']
data_ref['Inflow reverberatory']=DSM_refiner.i*data_ref['Inflow share reverberatory']
data_ref['Inflow induction']=DSM_refiner.i*data_ref['Inflow share induction']
data_ref['Inflow hydrogen']=DSM_refiner.i*data_ref['Inflow share hydrogen']

In [18]:
#export total data_rem
data_ref.to_excel('Python exports/Secondary_refiner' + '_' + scenario_name + ', L' + str(lifetime_furnaces) +'.xlsx')

# Foundry

In [19]:
#loading data_rem
data_foundry = pd.read_excel(file, sheet_name = 'Foundry', usecols='A:AQ', skiprows=1, nrows=151)
#Input data_rem (smelter capacity stocks) stays the same
data_foundry.reset_index(drop=True, inplace=True)
data_foundry.fillna(0, inplace=True)


In [20]:
#foundry DSM model


DSM_foundry = DynamicStockModel(t=data_foundry['year'],
                         s=data_foundry['Net capacity foundry [kt]'],
                         lt={'Type': 'Normal', 
                             'Mean': np.array([lifetime_furnaces]),
                             'StdDev': np.array([std_dev_furnaces]) 
                             }
                            )

DSM_foundry.compute_stock_driven_model() 
DSM_foundry.compute_outflow_total()

data_foundry['Inflow foundry'] = DSM_foundry.i
data_foundry['Outflow foundry'] = DSM_foundry.o

In [21]:
#calculating stock by cohort to calculate capacity per technology
stock_cohort_foundry = DSM_foundry.s_c
stock_cohort_foundry

array([[0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 3.48091918e-02, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 3.47884716e-02, 6.38861705e-03, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 5.50875997e-05, 1.01103634e-05, ...,
        8.62232873e+01, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 5.50875997e-05, 1.01103634e-05, ...,
        8.61719626e+01, 1.64549569e+02, 0.00000000e+00],
       [0.00000000e+00, 5.50875997e-05, 1.01103634e-05, ...,
        8.61037130e+01, 1.64451620e+02, 2.41191891e+02]])

In [22]:
#setting the inflow shares for 2023 onwards depending on scenario
if scenario_tech_name == 'no change':
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share shaft'] = data_foundry['Inflow share shaft'][122]
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share crucible fossil'] = data_foundry['Inflow share crucible fossil'][122]
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share induction'] = data_foundry['Inflow share induction'][122]
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share reverberatory'] = data_foundry['Inflow share reverberatory'][122]
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share hydrogen'] = data_foundry['Inflow share hydrogen'][122]

elif scenario_tech_name == 'high electricity':
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share shaft'] = 0
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share crucible fossil'] = 0
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share induction'] = 1
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share reverberatory'] = 0
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share hydrogen'] = 0
    
elif scenario_tech_name == 'electricity, hydrogen2030':
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share shaft'] = 0
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share crucible fossil'] = 0
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share induction'] = 1
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share reverberatory'] = 0
    data_foundry.loc[data_foundry['year']>=2023, 'Inflow share hydrogen'] = 0
    
    data_foundry.loc[data_foundry['year']>=2030, 'Inflow share induction'] = 0.5
    data_foundry.loc[data_foundry['year']>=2030, 'Inflow share hydrogen'] = 0.5

inflow_share_foundry_shaft=np.array([data_foundry['Inflow share shaft']])
inflow_share_foundry_crucible_fossil=np.array([data_foundry['Inflow share crucible fossil']])
inflow_share_foundry_induc=np.array([data_foundry['Inflow share induction']])
inflow_share_foundry_rev=np.array([data_foundry['Inflow share reverberatory']])
inflow_share_foundry_h2=np.array([data_foundry['Inflow share hydrogen']])

In [23]:
s_c_foundry_shaft = stock_cohort_foundry*inflow_share_foundry_shaft
s_c_foundry_crucible_fossil = stock_cohort_foundry*inflow_share_foundry_crucible_fossil
s_c_foundry_induc = stock_cohort_foundry*inflow_share_foundry_induc
s_c_foundry_rev = stock_cohort_foundry*inflow_share_foundry_rev
s_c_foundry_h2 = stock_cohort_foundry*inflow_share_foundry_h2



data_foundry['Net capacity shaft'] = np.sum(s_c_foundry_shaft, axis=1)
data_foundry['Net capacity crucible fossil'] = np.sum(s_c_foundry_crucible_fossil, axis=1)
data_foundry['Net capacity induction'] = np.sum(s_c_foundry_induc, axis=1)
data_foundry['Net capacity reverberatory'] = np.sum(s_c_foundry_rev, axis=1)
data_foundry['Net capacity hydrogen'] = np.sum(s_c_foundry_h2, axis=1)

In [24]:
data_foundry['Inflow shaft']=DSM_foundry.i*data_foundry['Inflow share shaft']
data_foundry['Inflow crucible fossil']=DSM_foundry.i*data_foundry['Inflow share crucible fossil']
data_foundry['Inflow reverberatory']=DSM_foundry.i*data_foundry['Inflow share reverberatory']
data_foundry['Inflow induction']=DSM_foundry.i*data_foundry['Inflow share induction']
data_foundry['Inflow hydrogen']=DSM_foundry.i*data_foundry['Inflow share hydrogen']

In [25]:
#export total data_rem
data_foundry.to_excel('Python exports/Secondary_foundry' + '_' + scenario_name + ', L' + str(lifetime_furnaces) +'.xlsx')